
# Fine-Tuning T5 for Machine Translation

### 1. Install and import dependencies

In [1]:
!pip -q install transformers datasets sentencepiece accelerate sacrebleu evaluate

In [1]:

import math
import os
import random
from dataclasses import dataclass
from typing import Dict, List

import torch
from torch.utils.data import DataLoader
from torch.nn.utils import clip_grad_norm_
from tqdm.auto import tqdm

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    T5ForConditionalGeneration,
    get_linear_schedule_with_warmup,
)

from sacrebleu import corpus_bleu
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

print(f"PyTorch: {torch.__version__}")
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device name:", torch.cuda.get_device_name(0))
    print("Compute capability may vary; mixed precision can speed up training.")
    

PyTorch: 2.10.0+cpu
CUDA available: False


### 2. Configuration & reproducibility

In [2]:

# ---- Experiment config ----
MODEL_NAME = "t5-small"           # ssmall version of T5 for faster training
SRC_LANG = "English"
TGT_LANG = "French"
MAX_SOURCE_LENGTH = 128
MAX_TARGET_LENGTH = 128
BATCH_SIZE = 8
NUM_EPOCHS = 2                     # keep small for speed
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06                # % of total steps used for LR warmup
GRAD_ACCUM_STEPS = 2               # simulate larger batch on limited GPU
MAX_NEW_TOKENS = 64                # generation length for eval/inference
NUM_BEAMS = 4                      # beam search width for generation

# Subsampling for fast classroom runs (set to None to use full sets)
SUBSET_TRAIN = 4000
SUBSET_VALID = 500

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using DEVICE:", DEVICE)

# Where to save figures/metrics
OUT_DIR = Path("t5_runs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
    

Using DEVICE: cpu


### 3. Load a parallel corpus for transalation (OPUS Books: English→French)

In this example we will be using the [*OPUS books*](https://huggingface.co/datasets/Helsinki-NLP/opus_books). This is a collection of copyright free books multilingually aligned for 16 languages. 

In [3]:
# We load the english-french portion of the OPUS Books dataset
raw_dset = load_dataset("opus_books", "en-fr")
print(raw_dset)

VALIDATION_RATIO = 0.2
split = raw_dset["train"].train_test_split(test_size=VALIDATION_RATIO, seed=SEED)
train_raw = split["train"]
valid_raw = split["test"]

# Apply subsampling if configured
if SUBSET_TRAIN is not None:
    train_raw = train_raw.shuffle(seed=SEED).select(range(min(SUBSET_TRAIN, len(train_raw))))
if SUBSET_VALID is not None:
    valid_raw = valid_raw.shuffle(seed=SEED).select(range(min(SUBSET_VALID, len(valid_raw))))

print("Train size:", len(train_raw), "| Valid size:", len(valid_raw))
    

DatasetDict({
    train: Dataset({
        features: ['id', 'translation'],
        num_rows: 127085
    })
})
Train size: 4000 | Valid size: 500



### 4. Tokenization & preprocessing

In T5, all tasks are frame as text-to-text. A task prefix is added before the input text, in this case, `"translate English to French:"`. The task prefix and the input text are tokenized using the T5 tokenizer (truncated to `MAX_SOURCE_LENGTH`). We also tokenize the target text for the decoder (truncated to `MAX_TARGET_LENGTH`).


In [4]:

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

prefix = f"translate {SRC_LANG} to {TGT_LANG}: "

def preprocess_examples(batch):
    src_texts = [prefix + ex["en"] for ex in batch["translation"]]
    tgt_texts = [ex["fr"] for ex in batch["translation"]]
    model_inputs = tokenizer(src_texts, max_length=MAX_SOURCE_LENGTH, truncation=True)
    labels = tokenizer(tgt_texts, max_length=MAX_TARGET_LENGTH, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Map with batched=True returns lists of lists (no padding yet)
train_tokenized = train_raw.map(preprocess_examples, batched=True, remove_columns=train_raw.column_names)
valid_tokenized = valid_raw.map(preprocess_examples, batched=True, remove_columns=valid_raw.column_names)

print(train_tokenized)
print(valid_tokenized)
    

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 4000
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 500
})


### 5. Data Loader
We pad to the longest sequence in each batch, both inputs and labels. In the labels, pad tokens are converted to **-100** so the loss ignores them.


In [5]:
def collate_fn(features):
    input_batch = {"input_ids": [f["input_ids"] for f in features],
                   "attention_mask": [f["attention_mask"] for f in features]}
    labels_batch = {"input_ids": [f["labels"] for f in features]}

    padded_inputs = tokenizer.pad(input_batch, padding=True, return_tensors="pt")
    padded_labels = tokenizer.pad(labels_batch, padding=True, return_tensors="pt")["input_ids"]
    # Replace padding token id with -100 to ignore in loss computation
    padded_labels = padded_labels.masked_fill(padded_labels == tokenizer.pad_token_id, -100)
    padded_inputs["labels"] = padded_labels
    return padded_inputs

train_loader = DataLoader(train_tokenized, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
valid_loader = DataLoader(valid_tokenized, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

len(train_loader), len(valid_loader)

(500, 63)

### 6. Model, optimizer, and scheduler

We will be using [`T5ForConditionalGeneration`](https://huggingface.co/docs/transformers/en/model_doc/t5#transformers.T5ForConditionalGeneration), for text-to-text generation. 

The specific checkpoint of the model is specified by the parametre `MODEL_NAME`. By default, we will use 't5-small', the smallest available GPT model with 60M parameters. You can also try larger versions 't5-base' or 't5-large' (see https://huggingface.co/collections/google/t5-release)

In [6]:
# We load the pretrained T5 model for conditional generation
model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)
model.to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

num_update_steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
t_total_steps = NUM_EPOCHS * num_update_steps_per_epoch
num_warmup_steps = int(WARMUP_RATIO * t_total_steps)

lr_scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=t_total_steps,
)

print(f"Total steps: {t_total_steps} | Warmup steps: {num_warmup_steps}")
    

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Total steps: 500 | Warmup steps: 30


### 7. Inference with the pre-trained model

We do inference for the first 10 elements in the validation set to check the performanca of the pre-trained model before fine-tuning.

In [9]:
model.eval()

# Build a batch from the first 10 validation samples for a forward pass
examples = valid_tokenized.select(range(10))
enc = tokenizer.pad(
    {
        "input_ids": [x["input_ids"] for x in examples],
        "attention_mask": [x["attention_mask"] for x in examples],
    },
    padding=True,
    return_tensors="pt",
)
enc = {k: v.to(DEVICE) for k, v in enc.items()}
gen = model.generate(
    input_ids=enc['input_ids'],
    attention_mask=enc['attention_mask'],
    max_new_tokens=MAX_NEW_TOKENS,
    num_beams=NUM_BEAMS,
)
decoded = tokenizer.batch_decode(gen, skip_special_tokens=True)

for text, pred in zip(examples, decoded):
    print("Source:", tokenizer.decode(text["input_ids"], skip_special_tokens=True))
    print("Reference:", tokenizer.decode(text["labels"], skip_special_tokens=True))
    print("Prediction:", pred)
    print("---")
    

Review: translate English to French: Therese then saw what a terrible shock her aunt had received.
Reference: Thérèse put voir quel terrible coup avait reçu sa tante.
Prediction: Therese a alors vu ce terrible choc que sa tante avait reçu.
---
Review: translate English to French: "Ah!" cried Neb, "if my master was here, he would know what to do!"
Reference: -- Ah! s'écria Nab, s'il était là, mon maître, il saurait bien vous en faire!»
Prediction: "Ah!" cria Neb, si mon maître était ici, il savait quoi faire!
---
Review: translate English to French: As for our neglect, our isolation in the depths of this cell, I was afraid to guess at how long it might last.
Reference: Quant à notre abandon, notre isolement au fond de cette cellule, je n'osais estimer ce qu'il pourrait durer.
Prediction: En ce qui concerne notre négligence, notre isolement dans les profondeurs de cette cellule, je craignais de deviner combien de temps elle pourrait durer.
---
Review: translate English to French: A gentl

### 8. Training loop


In [ ]:
best_valid_bleu = -1.0
train_loss_history, valid_loss_history, bleu_history = [], [], []

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    running_loss = 0.0
    pbar = tqdm(enumerate(train_loader, start=1), total=len(train_loader), desc=f"Epoch {epoch} [train]")

    optimizer.zero_grad(set_to_none=True)
    for step, batch in pbar:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}

        outputs = model(**batch)
        # Scales the loss before backpropagation so that gradient accumulation matches the magnitude of a larger effective batch
        loss = outputs.loss / GRAD_ACCUM_STEPS
        loss.backward()

        # Only update weights and step scheduler every GRAD_ACCUM_STEPS to simulate larger batch size
        if step % GRAD_ACCUM_STEPS == 0 or step == len(train_loader):
            clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            lr_scheduler.step()

       # For logging, we accumulate the loss scaled by GRAD_ACCUM_STEPS to reflect the effective batch size.   
        running_loss += loss.item() * GRAD_ACCUM_STEPS
        avg_loss = running_loss / step
        pbar.set_postfix({"loss": f"{avg_loss:.4f}"})

    # ---- Validation & BLEU after each epoch ----
    model.eval()
    gen_texts = []
    ref_texts = []
    val_loss_running = 0.0

    with torch.no_grad():
        pbar_val = tqdm(valid_loader, desc=f"Epoch {epoch} [valid]")
        for batch in pbar_val:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}

            
            outputs = model(**batch)
            val_loss_running += outputs.loss.item()

            # Generate translation using beam search with NUM_BEAMS
            # Check different options for sampling during generation in https://huggingface.co/docs/transformers/en/main_classes/text_generation
            generated_tokens = model.generate(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                max_new_tokens=MAX_NEW_TOKENS,
                num_beams=NUM_BEAMS,
            )

            decoded_preds = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)

            labels = batch["labels"].clone()
            # Replace back id -100 with the PAD token for evaluation
            labels[labels == -100] = tokenizer.pad_token_id
            decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

            gen_texts.extend([p.strip() for p in decoded_preds])
            ref_texts.extend([r.strip() for r in decoded_labels])

    bleu = corpus_bleu(gen_texts, [ref_texts])
    mean_val_loss = val_loss_running / max(1, len(valid_loader))

    print(f"Epoch {epoch}: train_loss={avg_loss:.4f} | valid_loss={mean_val_loss:.4f} | BLEU={bleu.score:.2f}")
    train_loss_history.append(avg_loss)
    valid_loss_history.append(mean_val_loss)
    bleu_history.append(bleu.score)

    np.save(OUT_DIR / "train_loss_history.npy", np.array(train_loss_history))
    np.save(OUT_DIR / "valid_loss_history.npy", np.array(valid_loss_history))
    np.save(OUT_DIR / "bleu_history.npy", np.array(bleu_history))

    if bleu.score > best_valid_bleu:
        best_valid_bleu = bleu.score
        save_dir = f"t5-{SRC_LANG[:2].lower()}2{TGT_LANG[:2].lower()}-best"
        os.makedirs(save_dir, exist_ok=True)
        model.save_pretrained(save_dir)
        tokenizer.save_pretrained(save_dir)
        print(f"Saved new best model to: {save_dir}")
    

Epoch 1 [train]:   0%|          | 0/500 [00:00<?, ?it/s]

KeyboardInterrupt: 


### 9. Plot metrics across epochs
We visualize **training/validation loss** and **BLEU** score.


In [ ]:

# Ensure histories exist (useful if you re-run only this cell)
if 'train_loss_history' not in globals():
    train_loss_history = np.load(OUT_DIR / 'train_loss_history.npy').tolist()
    valid_loss_history = np.load(OUT_DIR / 'valid_loss_history.npy').tolist()
    bleu_history = np.load(OUT_DIR / 'bleu_history.npy').tolist()

epochs = list(range(1, len(train_loss_history) + 1))

# --- Plot 1: Loss ---
plt.figure(figsize=(6, 4))
plt.plot(epochs, train_loss_history, label='Training loss')
plt.plot(epochs, valid_loss_history, label='Validation loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training vs Validation Loss')
plt.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / 'loss_plot.png', dpi=150)
plt.show()

# --- Plot 2: BLEU ---
plt.figure(figsize=(6, 4))
plt.plot(epochs, bleu_history, label='Validation BLEU')
plt.xlabel('Epoch')
plt.ylabel('BLEU')
plt.title('Validation BLEU')
plt.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / 'bleu_plot.png', dpi=150)
plt.show()

print('Saved figures to:', OUT_DIR.resolve())
    

### 10. Inference with the fine-tuned model

We do inference for the first 10 elements in the validation set to check the performanca of the fine-tuned model.

In [ ]:
model.eval()

# Build a batch from the first 10 validation samples for a forward pass
examples = valid_tokenized.select(range(10))
enc = tokenizer.pad(
    {
        "input_ids": [x["input_ids"] for x in examples],
        "attention_mask": [x["attention_mask"] for x in examples],
    },
    padding=True,
    return_tensors="pt",
)
enc = {k: v.to(DEVICE) for k, v in enc.items()}
gen = model.generate(
    input_ids=enc['input_ids'],
    attention_mask=enc['attention_mask'],
    max_new_tokens=MAX_NEW_TOKENS,
    num_beams=NUM_BEAMS,
)
decoded = tokenizer.batch_decode(gen, skip_special_tokens=True)

for text, pred in zip(examples, decoded):
    print("Source:", tokenizer.decode(text["input_ids"], skip_special_tokens=True))
    print("Reference:", tokenizer.decode(text["labels"], skip_special_tokens=True))
    print("Prediction:", pred)
    print("---")
    

Review: translate English to French: Therese then saw what a terrible shock her aunt had received.
Reference: Thérèse put voir quel terrible coup avait reçu sa tante.
Prediction: Therese a alors vu ce terrible choc que sa tante avait reçu.
---
Review: translate English to French: "Ah!" cried Neb, "if my master was here, he would know what to do!"
Reference: -- Ah! s'écria Nab, s'il était là, mon maître, il saurait bien vous en faire!»
Prediction: "Ah!" cria Neb, si mon maître était ici, il savait quoi faire!
---
Review: translate English to French: As for our neglect, our isolation in the depths of this cell, I was afraid to guess at how long it might last.
Reference: Quant à notre abandon, notre isolement au fond de cette cellule, je n'osais estimer ce qu'il pourrait durer.
Prediction: En ce qui concerne notre négligence, notre isolement dans les profondeurs de cette cellule, je craignais de deviner combien de temps elle pourrait durer.
---
Review: translate English to French: A gentl